# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [18]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* :-  The line chart uses 1970 as the starting point (index = 100) to show how each continent’s population has grown over time. This makes it easier to compare growth across continents, even though Asia has billions of people while Oceania has only millions.

Africa is highlighted in colour because it is the main focus, while the other continents are shown in grey for comparison. Each continent is labelled directly at the end of its line instead of using a separate legend. The three labels that appear close together are slightly spaced out to make them easier to read.

In [19]:
# Q1
GREY, ACCENT = '#BDBDBD', '#0072B2'

cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont[cont.Year == 1970].set_index('Continent')['Population']
cont['Index'] = cont['Population'] / cont['Continent'].map(base) * 100   # 1970 = 100

last = cont[cont.Year == 2022].set_index('Continent')['Index']
second = last.drop('Africa').sort_values(ascending=False)
title = (f"<b>Africa's population is {last['Africa']/100:.1f}× its 1970 size</b> — "
         f"the next-fastest continent, {second.index[0]}, is only {second.iloc[0]/100:.1f}×")

fig = px.line(cont, x='Year', y='Index', color='Continent', template='plotly_white',
              color_discrete_map={c: (ACCENT if c == 'Africa' else GREY) for c in cont.Continent.unique()})
fig.update_traces(line_width=2)
fig.update_traces(line_width=4, selector=dict(name='Africa'))

nudge = {'Oceania': 11, 'South America': 0, 'Asia': -11}   # keep bunched labels apart
for c, v in last.items():
    fig.add_annotation(x=2022, y=v, xanchor='left', xshift=8, yshift=nudge.get(c, 0), showarrow=False,
                       text=f"<b>{c} {v/100:.1f}×</b>" if c == 'Africa' else f"{c} {v/100:.1f}×",
                       font=dict(color=ACCENT if c == 'Africa' else '#555', size=13 if c == 'Africa' else 12))

fig.add_hline(y=100, line_dash='dot', line_color='#888', line_width=1)
fig.update_layout(title=dict(text=title, x=0, font_size=18), showlegend=False,
                  margin=dict(l=60, r=150, t=80, b=50), height=520, width=900)
fig.update_xaxes(title=None, tickvals=[1970, 1980, 1990, 2000, 2010, 2022], showgrid=False, range=[1969, 2023])
fig.update_yaxes(title='Population, 1970 = 100', gridcolor='#EEE', zeroline=False)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* :- Sorted horizontal bar chart. This chart compares 15 countries based on the number of people added. The countries are arranged from highest to lowest, making it easy to see which ones contributed the most. Horizontal bars are used because they make long country names easier to read. Africa is highlighted in a different colour because the main focus is to identify how many of these countries are in Africa. The total count is included in the title, so readers don't have to count the bars themselves. Values are also displayed directly on the bars, which makes an x-axis and gridlines unnecessary.

In [20]:
# Q2
GREY, ACCENT = '#BDBDBD', '#0072B2'

d = df.assign(Added=(df['2022 Population'] - df['1970 Population']) / 1e6)
d = d.nlargest(15, 'Added').sort_values('Added')
n_africa = (d.Continent == 'Africa').sum()
top2 = d.nlargest(2, 'Added')
title = (f"<b>{n_africa} of the 15 biggest population gains are African</b> — "
         f"yet India and China alone added {top2.Added.sum()/1e3:.2f} billion people")

fig = px.bar(d, x='Added', y='Country', orientation='h', template='plotly_white')
fig.update_traces(marker_color=[ACCENT if c == 'Africa' else GREY for c in d.Continent],
                  texttemplate='%{x:,.0f}M', textposition='outside', cliponaxis=False, width=0.7)
fig.add_annotation(xref='paper', yref='paper', x=0.98, y=0.12, xanchor='right', showarrow=False, align='right',
                   text=f"<span style='color:{ACCENT}'>■</span> Africa<br><span style='color:{GREY}'>■</span> Rest of world",
                   font=dict(size=13, color='#444'))
fig.update_layout(title=dict(text=title + "<br><sup>People added between 1970 and 2022, millions</sup>", x=0, font_size=18),
                  showlegend=False, margin=dict(l=130, r=60, t=90, b=30), height=560, width=900)
fig.update_xaxes(visible=False, range=[0, d.Added.max() * 1.12])
fig.update_yaxes(title=None, ticks='', tickfont_size=13)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* :- Scatter plot with log–log axes. A scatter plot is a good way to show the relationship between two numerical variables, compare countries across continents, and identify unusual values. Since both variables have a wide range of values, using normal scales would make most countries appear squeezed into one corner because of large countries like Russia and China. Logarithmic scales help spread the data points out and make the overall trend easier to see. Different colours are used for each continent, with a colour-blind-friendly palette. A single dashed line shows the general trend, and only the countries that are far away from this trend are labelled to keep the chart clean and easy to understand.

In [21]:
# Q3
CONT = {'Asia': '#0072B2', 'Africa': '#D55E00', 'Europe': '#56B4E9',
        'North America': '#E69F00', 'South America': '#CC79A7', 'Oceania': '#999999'}   # Okabe-Ito, CVD-safe

lx, ly = np.log10(df['Area']), np.log10(df['2022 Population'])
slope, icpt = np.polyfit(lx, ly, 1)
r = np.corrcoef(lx, ly)[0, 1]
rc = {c: np.corrcoef(lx[g.index], ly[g.index])[0, 1] for c, g in df.groupby('Continent')}
weakest = min(rc, key=rc.get)
title = (f"<b>Bigger countries do have more people (r = {r:.2f} on log scales)</b> —<br>"
         f"but Greenland and the Falklands are near-empty, and city-states pack millions into tiny areas"
         f"<br><sup>The link is loosest in {weakest} (r = {rc[weakest]:.2f}) and tightest in Oceania and South America (r = {rc['Oceania']:.2f})</sup>")

fig = px.scatter(df, x='Area', y='2022 Population', color='Continent', log_x=True, log_y=True,
                 hover_name='Country', template='plotly_white', color_discrete_map=CONT,
                 category_orders={'Continent': list(CONT)})
fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=0.8, color='white')))

xs = np.array([lx.min(), lx.max()])
fig.add_scatter(x=10 ** xs, y=10 ** (icpt + slope * xs), mode='lines', hoverinfo='skip', showlegend=False,
                line=dict(color='#444', width=1.5, dash='dash'))

# label only the countries that break the pattern (offsets in pixels)
labels = {'Greenland': (45, 25), 'Falkland Islands': (50, 25), 'Canada': (45, 30), 'Russia': (30, -35),
          'Australia': (-60, 30), 'China': (20, -35), 'India': (-40, -30), 'Bangladesh': (-55, -12),
          'Singapore': (45, -12), 'Macau': (-40, -25)}
for name, (ax, ay) in labels.items():
    row = df[df.Country == name].iloc[0]
    fig.add_annotation(x=np.log10(row.Area), y=np.log10(row['2022 Population']), text=name, ax=ax, ay=ay,
                       showarrow=True, arrowwidth=0.8, arrowcolor='#888', font=dict(size=12, color='#333'))

fig.update_layout(title=dict(text=title, x=0, font_size=17),
                  legend=dict(title=None, orientation='h', y=1.02, x=0, yanchor='bottom'),
                  margin=dict(l=70, r=30, t=150, b=60), height=640, width=950)
fig.update_xaxes(title='Land area (km², log scale)', showgrid=False,
                 tickvals=[10, 100, 1e3, 1e4, 1e5, 1e6, 1e7], ticktext=['10', '100', '1K', '10K', '100K', '1M', '10M'])
fig.update_yaxes(title='Population 2022 (log scale)', gridcolor='#EEE',
                 tickvals=[1e3, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9], ticktext=['1K', '10K', '100K', '1M', '10M', '100M', '1B'])
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*  Chart choice and justification: Box plot with individual country dots and a logarithmic y-axis.

I chose a box plot because the aim is to compare how population density varies within and across continents, rather than just looking at the average. It shows the median, quartiles and outliers for each continent, making it easier to understand both the typical density and the overall spread. The individual dots represent each country, so we can also see how many countries are included and how much they differ from one another.

Population density ranges from around 0.03 to 23,000 people per km², so I used a logarithmic scale to make the differences easier to see. Without it, most of the boxes would appear almost flat at the bottom of the chart. The continents are arranged from highest to lowest median density. North America has the highest median (221), so it is highlighted in a different colour while the remaining continents are shown in grey. Median values are displayed next to each box, and only countries with extreme values, such as Macau, Monaco, Singapore and Greenland, are labelled to keep the chart clear and easy to read.

In [22]:
# Q4
GREY, ACCENT = '#BDBDBD', '#0072B2'

med = df.groupby('Continent')['Density'].median().sort_values(ascending=False)
order = list(med.index)
top = order[0]
spread = {c: g.Density.max() / g.Density.min() for c, g in df.groupby('Continent')}
widest = max(spread, key=spread.get)
title = (f"<b>Spread within a continent dwarfs the gaps between them</b>:<br>typical (median) density is only "
         f"{med.min():.0f}–{med.max():.0f} people/km², but {widest} spans a {spread[widest]:,.0f}× range"
         f"<br><sup>Population density 2022, one dot per country or territory; {top} has the highest median, {order[-1]} the lowest</sup>")

fig = px.box(df, x='Continent', y='Density', color='Continent', points='all', log_y=True,
             hover_name='Country', category_orders={'Continent': order}, template='plotly_white',
             color_discrete_map={c: (ACCENT if c == top else GREY) for c in order})
fig.update_traces(boxmean=False, jitter=0.5, pointpos=0, marker=dict(size=5, opacity=0.55), line_width=1.5, width=0.55)

for c, v in med.items():   # median labels
    fig.add_annotation(x=c, y=np.log10(v), text=f"<b>{v:.0f}</b>", showarrow=False, xanchor='left', xshift=22,
                       font=dict(size=12, color=ACCENT if c == top else '#555'))
for name, (ax, ay) in {'Macau': (50, 8), 'Monaco': (50, 8), 'Singapore': (55, 10), 'Greenland': (50, 0)}.items():
    row = df[df.Country == name].iloc[0]
    fig.add_annotation(x=row.Continent, y=np.log10(row.Density), text=name, ax=ax, ay=ay, showarrow=True,
                       arrowwidth=0.8, arrowcolor='#888', font=dict(size=11, color='#333'))

fig.update_layout(title=dict(text=title, x=0, font_size=17), showlegend=False,
                  margin=dict(l=80, r=30, t=130, b=50), height=620, width=950)
fig.update_xaxes(title=None, showgrid=False)
fig.update_yaxes(title='People per km² (log scale)', gridcolor='#EEE',
                 tickvals=[0.1, 1, 10, 100, 1e3, 1e4], ticktext=['0.1', '1', '10', '100', '1,000', '10,000'])
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* :- Choropleth map with a diverging colour scale. A map is the best way to show which countries experienced population growth and which ones saw a decline. Since the data includes both positive and negative percentage changes, different colours are used to make the difference clear. Blue represents population growth, orange represents population decline, and grey indicates no change (0%). These colours are also easier to distinguish for people with red-green colour blindness. The colour scale is limited to +150% so that countries with extremely high growth, such as Qatar, don't make the differences between other countries difficult to see. Percentage change is used instead of actual population numbers so that both small and large countries can be compared fairly.

In [23]:
# Q5
d = df.assign(Change=(df['2022 Population'] / df['2000 Population'] - 1) * 100)
n_shrink = (d.Change < 0).sum()
n_eur = ((d.Change < 0) & (d.Continent == 'Europe')).sum()
title = (f"<b>Only {n_shrink} of {len(d)} countries shrank since 2000</b> —<br>"
         f"{n_eur} of them in Europe (Baltics, Balkans, Eastern Europe), while every African country grew"
         f"<br><sup>Population change 2000–2022, %</sup>")

lo, hi = -40, 150                       # colour range; 0 sits at (0-lo)/(hi-lo) of the scale
mid = (0 - lo) / (hi - lo)
scale = [[0, '#B35806'], [mid / 2, '#F1A340'], [mid, '#F0F0F0'], [mid + (1 - mid) / 2, '#6BAED6'], [1, '#08519C']]

fig = px.choropleth(d, locations='CCA3', color='Change', hover_name='Country', range_color=[lo, hi],
                    color_continuous_scale=scale, template='plotly_white', projection='natural earth')
fig.update_geos(showframe=False, showcoastlines=False, showcountries=True, countrycolor='white',
                showland=True, landcolor='#F7F7F7', showocean=False, lakecolor='white')
fig.update_layout(title=dict(text=title, x=0, font_size=17), margin=dict(l=10, r=10, t=120, b=10), height=580, width=1000,
                  coloraxis_colorbar=dict(title=None, tickvals=[-40, 0, 50, 100, 150],
                                          ticktext=['−40%', '0', '+50%', '+100%', '≥ +150%'],
                                          len=0.6, thickness=14, outlinewidth=0))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* :- Treemap. A treemap is a good choice because it shows how the world's population is distributed across continents and the countries within them in a single chart. Each rectangle represents a country's population, and its size shows how much it contributes to the world total. The grouping of rectangles makes it easy to see which countries belong to each continent. Using a bar chart for around 230 countries would make the information too crowded and difficult to read. Different colours are used to represent each continent, while the labels show each country's share of the global population. Smaller rectangles without enough space for labels are left without text, but their details can still be viewed by hovering over them. Although comparing rectangle sizes is not as precise as comparing bar lengths, a treemap works well here because the main goal is to understand the overall distribution and identify the largest population groups.

In [24]:
# Q6
CONT = {'Asia': '#0072B2', 'Africa': '#D55E00', 'Europe': '#56B4E9',
        'North America': '#E69F00', 'South America': '#CC79A7', 'Oceania': '#999999', '(?)': 'white'}

pop = df.set_index('Country')['2022 Population']
total = pop.sum()
asia = df[df.Continent == 'Asia']['2022 Population'].sum() / total
ic = (pop['India'] + pop['China']) / total
title = (f"<b>Asia holds {asia:.0%} of the world's {total/1e9:.1f} billion people</b> —<br>"
         f"India and China alone account for {ic:.0%}"
         f"<br><sup>Area = 2022 population; labels show share of world total</sup>")

fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'], values='2022 Population',
                 color='Continent', color_discrete_map=CONT, template='plotly_white')
fig.update_traces(root_color='white', texttemplate='%{label}<br>%{percentRoot:.1%}', textfont_size=13,
                  marker=dict(line=dict(width=1.5, color='white')),
                  hovertemplate='<b>%{label}</b><br>%{value:,.0f} people<br>%{percentRoot:.1%} of world<extra></extra>')
fig.update_layout(title=dict(text=title, x=0, font_size=17), margin=dict(l=10, r=10, t=120, b=10), height=660, width=1000,
                  uniformtext=dict(minsize=10, mode='hide'))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* :- 100% stacked bar chart. This chart is useful because it shows how each continent's share of the world's total population changes over time. Since the total always adds up to 100%, it becomes easier to compare how the proportions shift between years. Africa is highlighted in blue because its share is increasing, while Europe is shown in orange because its share is decreasing. Asia and the rest of the world are shown in grey to keep the focus on the main changes. Each section is labelled with its percentage, and the continent names are placed directly on the right side of the chart instead of using a separate legend. Since the years in the dataset are not evenly spaced (2015, 2020 and 2022), the x-axis treats them as separate categories rather than showing them on a continuous time scale.

In [25]:
# Q7
COL = {'Africa': '#0072B2', 'Asia': '#8C8C8C', 'Europe': '#E69F00', 'Rest of the world': '#D9D9D9'}

s = df_long.assign(Group=df_long.Continent.where(df_long.Continent.isin(['Africa', 'Asia', 'Europe']), 'Rest of the world'))
s = s.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
s['Share'] = s['Population'] / s.groupby('Year')['Population'].transform('sum') * 100
s['Year'] = s['Year'].astype(str)
s['Label'] = s['Share'].map(lambda v: f"{v:.1f}%")

p = s.pivot(index='Year', columns='Group', values='Share')
title = (f"<b>Africa's share of world population nearly doubled ({p.loc['1970','Africa']:.1f}% → {p.loc['2022','Africa']:.1f}%)</b> "
         f"while Europe's almost halved ({p.loc['1970','Europe']:.1f}% → {p.loc['2022','Europe']:.1f}%)"
         f"<br><sup>Asia stayed at roughly {p.loc['1970','Asia']:.0f}–{p['Asia'].max():.0f}% of the world throughout</sup>")

fig = px.bar(s, x='Year', y='Share', color='Group', text='Label', template='plotly_white', color_discrete_map=COL,
             category_orders={'Group': ['Africa', 'Asia', 'Europe', 'Rest of the world'],
                              'Year': sorted(s.Year.unique())})
fig.update_traces(textposition='inside', insidetextanchor='middle', marker_line=dict(width=2, color='white'),
                  textfont=dict(size=12))
fig.update_traces(textfont_color='white', selector=dict(name='Africa'))
fig.update_traces(textfont_color='#222', selector=lambda t: t.name != 'Africa')

cum = 0   # group names at the right edge, aligned with the 2022 segments
for g in ['Africa', 'Asia', 'Europe', 'Rest of the world']:
    v = p.loc['2022', g]
    fig.add_annotation(xref='paper', x=1.0, yref='y', y=cum + v / 2, xanchor='left', xshift=8, showarrow=False,
                       text=f"<b>{g}</b>" if g in ('Africa', 'Europe') else g,
                       font=dict(size=13, color=COL[g] if g in ('Africa', 'Europe') else '#555'))
    cum += v

fig.update_layout(title=dict(text=title, x=0, font_size=17), showlegend=False, bargap=0.2,
                  margin=dict(l=40, r=130, t=110, b=40), height=600, width=950)
fig.update_xaxes(title=None, type='category')
fig.update_yaxes(visible=False, range=[0, 100])
fig.show()